# CropHist Iowa Sample

This notebook uses the published free Iowa technical sample.

Goal:

- fetch a small batch of Iowa field histories through the CropHist API
- reshape the histories into an analyst-friendly table
- visualize rotation patterns
- produce a simple 2026 baseline forecast

The field IDs below come from the published Iowa sample export generated on 2026-04-06.

In [ ]:
import os
from itertools import islice

import matplotlib.pyplot as plt
import pandas as pd
import requests

API_BASE = os.environ.get('CROPHIST_API_BASE', 'https://api.crophist.com')
API_KEY = os.environ.get('CROPHIST_API_KEY', 'replace-me')

# These IDs come from the published Iowa sample export.
FIELD_IDS = [
    '5003.VJV5',
    '5003.VW5D',
    '5003.YRQW',
    '5003.Z5MQ',
]

assert API_KEY != 'replace-me', 'Set CROPHIST_API_KEY before running the live API calls.'

In [ ]:
def chunks(items, size):
    iterator = iter(items)
    while True:
        batch = list(islice(iterator, size))
        if not batch:
            return
        yield batch


def fetch_histories(field_ids):
    rows = []
    headers = {
        'Authorization': f'Bearer {API_KEY}',
        'Content-Type': 'application/json',
    }
    for batch in chunks(field_ids, 100):
        response = requests.post(
            f'{API_BASE}/v1/fields',
            headers=headers,
            json={'field_ids': batch},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        rows.extend(payload['results'])
    return rows

In [ ]:
results = fetch_histories(FIELD_IDS)

flat_rows = []
for item in results:
    field_id = item['field_id']
    for entry in item.get('history', []):
        flat_rows.append({
            'field_id': field_id,
            'year': entry['year'],
            'crop_code': entry['crop_code'],
            'crop_name': entry['crop_name'],
            'fraction': entry.get('fraction'),
        })

history_df = pd.DataFrame(flat_rows).sort_values(['field_id', 'year'])
history_df.head()

In [ ]:
pivot = history_df.pivot(index='year', columns='field_id', values='crop_name')
display(pivot.tail(8))

example_field = history_df['field_id'].iloc[0]
example = history_df[history_df['field_id'] == example_field].copy()

plt.figure(figsize=(10, 4))
plt.plot(example['year'], example['crop_code'], marker='o')
plt.title(f'Crop sequence for {example_field}')
plt.xlabel('Year')
plt.ylabel('Crop code')
plt.grid(alpha=0.2)
plt.show()

In [ ]:
rotation_summary = (
    history_df
    .assign(is_corn_soy=lambda df: df['crop_name'].isin(['Corn', 'Soybeans']))
    .groupby('field_id')
    .agg(
        years=('year', 'count'),
        corn_soy_share=('is_corn_soy', 'mean'),
        last_crop=('crop_name', 'last'),
    )
    .reset_index()
)

rotation_summary.sort_values('corn_soy_share', ascending=False).head()

In [ ]:
def baseline_forecast(last_crop):
    if last_crop == 'Corn':
        return 'Soybeans'
    if last_crop == 'Soybeans':
        return 'Corn'
    return last_crop


rotation_summary['forecast_2026'] = rotation_summary['last_crop'].map(baseline_forecast)
forecast_mix = rotation_summary['forecast_2026'].value_counts(normalize=True).rename('share')
forecast_mix.head(10)